# P2 — Task 2 — Financial Health

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thanh-tai-435/BI10GroupYAPPERS/blob/main/notebooks/task2_financial_health.ipynb)

**Slide sở hữu:** 8, 9, 10 · **Slide mới:** +1: Khác biệt theo nghề / tuổi / tỉnh  
**Đã có:** task2.py, task2_findings.md, chart t2_fhs_distribution / t2_low_health_drivers / t2_crossover

**3 luật:** (1) Wellbeing, KHÔNG credit scoring — FHS không bao giờ dùng để từ chối/giảm hạn mức/khoá. (2) So bằng tỷ lệ, không so VND tuyệt đối. (3) Mỗi nhận định có số hoặc chart.

Hướng dẫn chi tiết: `draft/YAPPERS_huong_dan_Colab.xlsx`, sheet của bạn.

In [ ]:
#@title ⚙️ SETUP — chạy 1 lần mỗi phiên Colab (không cần pip install gì cả)
# Colab có sẵn pandas / numpy / scikit-learn / matplotlib / openpyxl -> đủ cho mọi script trong draft/.
import os, sys, gzip, shutil
DRIVE_DATA = "/content/drive/MyDrive/BI10_ROUND01_DATASET"   # thư mục data trên Drive (xem sheet 'Setup Colab')
REPO = "/content/BI10GroupYAPPERS"

from google.colab import drive, userdata
drive.mount("/content/drive")

if os.path.exists(REPO):
    os.system(f"git -C {REPO} pull -q")
else:
    tok = userdata.get("GH_TOKEN")   # 🔑 Secrets bên trái: tên GH_TOKEN, bật "Notebook access"
    os.system(f"git clone -q https://{tok}@github.com/thanh-tai-435/BI10GroupYAPPERS.git {REPO}")
assert os.path.exists(REPO), "Clone thất bại: kiểm tra GH_TOKEN + đã được mời vào repo chưa"
os.chdir(REPO)

# Dựng lại đúng đường dẫn mà các script draft/*.py đang dùng
D = "BI10_ROUND01_DATASET"
TX = f"{D}/consumer_transactions_2025.parquet/consumer_transactions_2025.csv"
os.makedirs(os.path.dirname(TX), exist_ok=True)
for f in ["consumer_financial_health_engagement_2025.csv", "data_dictionary.xlsx"]:
    if not os.path.exists(f"{D}/{f}") and os.path.exists(f"{DRIVE_DATA}/{f}"):
        shutil.copy(f"{DRIVE_DATA}/{f}", f"{D}/{f}")
if not os.path.exists(TX):
    gz, raw = f"{DRIVE_DATA}/consumer_transactions_2025.parquet.gz", f"{DRIVE_DATA}/consumer_transactions_2025.parquet/consumer_transactions_2025.csv"
    if os.path.exists(gz):   # .parquet.gz thực chất là CSV giao dịch nén gzip (134MB -> 597MB, ~30s)
        with gzip.open(gz) as src, open(TX, "wb") as dst: shutil.copyfileobj(src, dst)
    else:                    # hoặc upload nguyên thư mục .parquet/ chứa CSV 597MB
        shutil.copy(raw, TX)

# Script gốc gọi sys.stdout.reconfigure(); stdout của Colab không có hàm này -> cho thành no-op
if not hasattr(sys.stdout, "reconfigure"):
    type(sys.stdout).reconfigure = lambda self, **k: None
os.makedirs("outputs/figures", exist_ok=True)
if not os.path.exists("draft/task4_features.csv"):   # nhãn segment (git-ignore) — task3/task5/charts cần
    os.system("python draft/task4.py > /dev/null")
print("✅ Sẵn sàng — cwd:", os.getcwd())


## Bước 0 — Chạy baseline
Nạp toàn bộ biến của script gốc (`mon`, `cons`, `txn`…) vào notebook. So kết quả với `draft/task2_output.txt`.

In [ ]:
%run -i draft/task2.py

## Việc 1 — Deliverable 3 — khác biệt theo tỉnh / nghề / tuổi (SLIDE MỚI)  `[BẮT BUỘC · hạn 27/09 12:00]`
Tỉnh: dot plot FHS TB ± CI95% cho tỉnh ≥15 khách + đường TB toàn quốc. Nghề: 396 nghề quá vụn → gom 6–8 nhóm bằng từ khoá tiếng Việt (sinh viên/học sinh, kinh doanh, nhân viên/văn phòng, công nhân/lao động, chuyên môn: giáo viên/bác sĩ/kỹ sư, hưu trí, khác) — dict mapping ghi rõ trong code. Tuổi: dùng bảng CI của P1. Kết luận mong đợi: nhân khẩu chênh ít → nhắm theo tỷ lệ hành vi.

**Giao:** Chart t2_demographics.png + bullet slide mới

In [ ]:
# Gợi ý:
# MAP={'sinh viên':'Học sinh/SV','học sinh':'Học sinh/SV','kinh doanh':'Kinh doanh',...}
# cons['occ_grp']=cons.occupation.str.lower().map(lambda s: next((v for k,v in MAP.items() if k in s),'Khác'))


## Việc 2 — Dùng file giao dịch: thứ trong tuần / giờ  `[Cao · hạn 27/09 12:00]`
So tháng stressed (FHS<40) vs healthy (FHS≥80): tỷ trọng chi tiêu theo transaction_day_of_week, transaction_hour, và discretionary category. Nếu stress chi nhiều cuối tuần / đêm khuya → đó là THỜI ĐIỂM gửi Spend Alert → báo P5.

**Giao:** Heatmap ngày×giờ + 1 câu gửi P5

In [ ]:
# Gợi ý:
# t=txn.merge(mon[['consumer_id','month','financial_health_score']], left_on=['consumer_id','transaction_month'], right_on=['consumer_id','month'])


## Việc 3 — Crossover — bảng độ nhạy luật 43 khách  `[Cao · hạn 27/09 18:00]`
Số khách / số tháng / % tổng stress episode khi đổi ngưỡng FHS <35/<40/<45 × engagement ≥p70/p75/p80. Chứng minh 43 khách không ngẫu nhiên. Bảng nhỏ lên slide 10.

**Giao:** Bảng 3×3 + báo P5 nếu 43 / 52% đổi

In [ ]:
# Gợi ý:
# for f in (35,40,45):
#   for q in (.70,.75,.80): ...


## Việc 4 — FHS theo quintile spend_to_income  `[Trung bình · hạn 27/09 18:00]`
Bar chart FHS TB theo 5 quintile spend_to_income — trực quan hơn r = −0.90. Giữ caveat: FHS và ratio dùng chung trường chi tiêu → định hướng, không nhân quả.

**Giao:** Chart + bullet slide 9

In [ ]:
# Gợi ý:
# cons.groupby(pd.qcut(cons.spend_to_income_ratio,5)).fhs.mean()


## Findings mới (2–4 dòng / việc, có số)
- ...

---
**Lưu bài:** `File → Save a copy in GitHub` → repo `thanh-tai-435/BI10GroupYAPPERS`, branch `main`, path `notebooks/task2_financial_health.ipynb` (mỗi người chỉ lưu notebook của mình → không conflict). Chart: `savefig('outputs/figures/tN_ten.png')` rồi tải về gửi P5.